# Final-evaluation rerun: Ft Transformer

This notebook reruns the established tuning procedure on the **new frozen development split** created by notebook 10. It never loads the locked final-test rows. It selects validation thresholds for 70%, 75%, 80%, 85%, and 90% target recall, then saves the frozen model and artifacts for notebook 17.

# FT-Transformer Model Optimisation

This notebook evaluates an **FT-Transformer (FTT)** for 30-day hospital readmission prediction.

The goal is to make the comparison with the previous tuned models as fair as possible:

1. Use the same cleaned dataset and the same 18 predictive features.
2. Keep the final split at **60% model training / 20% threshold validation / 20% final test**.
3. Select model settings without touching the threshold-validation or final-test sets.
4. Select the probability threshold only on the 20% validation set.
5. Require validation recall of at least **80%**, then choose the eligible threshold with the lowest false-positive rate.
6. Evaluate the fixed model and fixed threshold once on the untouched final test set.

FT-Transformer differs from the tree models because each original feature is first converted into a learned token. A Transformer then uses self-attention to learn how the feature tokens relate to one another.

The implementation uses the official `rtdl_revisiting_models` package from the paper *Revisiting Deep Learning Models for Tabular Data*.

> **Computational note:** FT-Transformer is much more expensive to tune than the tree models. Run the notebook first with the smoke-test settings (`ARCHITECTURE_SEARCH_ITERATIONS = 2` and `TRAINING_SEARCH_ITERATIONS = 3`). If everything works, increase them to the suggested final values.

In [1]:
from pathlib import Path
from copy import deepcopy
import json
import random
import time

import numpy as np
import pandas as pd
import sklearn
import torch
import torch.nn as nn

from torch.utils.data import DataLoader, TensorDataset

from rtdl_revisiting_models import FTTransformer

from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    fbeta_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve
)
from sklearn.model_selection import (
    ParameterSampler,
    StratifiedKFold,
    train_test_split
)
from sklearn.preprocessing import StandardScaler

print("PyTorch version:", torch.__version__)
print("scikit-learn version:", sklearn.__version__)

PyTorch version: 2.13.0
scikit-learn version: 1.4.2


## 1. Reproducibility and compute device

The notebook automatically uses:

- CUDA if an NVIDIA GPU is available;
- Apple Metal (MPS) on a supported Apple Silicon Mac;
- otherwise CPU.

Neural-network training is stochastic, so fixed random seeds are used. Exact bit-for-bit reproducibility can still vary across hardware.

In [2]:
RANDOM_SEED = 42

def seed_everything(seed=RANDOM_SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything()

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif (
    hasattr(torch.backends, "mps")
    and torch.backends.mps.is_available()
):
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print("Training device:", DEVICE)

Training device: mps


## 2. Load the cleaned dataset and create the output folder

In [3]:
PROJECT_ROOT = Path.cwd()

for parent in [PROJECT_ROOT] + list(PROJECT_ROOT.parents):
    candidate = (
        parent
        / "Processed_Dataset"
        / "diabetic_data_cleaned_stage1.csv"
    )

    if candidate.exists():
        DATA_PATH = candidate
        PROJECT_ROOT = parent
        break
else:
    raise FileNotFoundError(
        "Could not find "
        "Processed_Dataset/diabetic_data_cleaned_stage1.csv"
    )

OUTPUT_DIR = (
    PROJECT_ROOT
    / "Model_Results"
    / "ft_transformer_optimisation"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

df = pd.read_csv(DATA_PATH)

print("Dataset path:")
print(DATA_PATH)

print("\nDataset shape:")
print(df.shape)

df.head()

# Final-evaluation artifacts are kept separate from the earlier development runs.
FINAL_EVALUATION_DIR = PROJECT_ROOT / "Final_Evaluation"
SPLIT_DIR = FINAL_EVALUATION_DIR / "Data_Splits"
OUTPUT_DIR = FINAL_EVALUATION_DIR / "Model_Artifacts" / "ft_transformer"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("\nFinal-evaluation output directory:")
print(OUTPUT_DIR)


Dataset path:
/Users/xiaohanmu/Desktop/School/UoB/Summer Individual Project/Processing_Pipeline/Pipeline_Selection/Processed_Dataset/diabetic_data_cleaned_stage1.csv

Dataset shape:
(69987, 56)

Final-evaluation output directory:
/Users/xiaohanmu/Desktop/School/UoB/Summer Individual Project/Processing_Pipeline/Pipeline_Selection/Final_Evaluation/Model_Artifacts/ft_transformer


## 3. Use the same 18 predictors as the other tuned models

FT-Transformer receives numerical and categorical variables separately, but the **information available to the model is unchanged**.

This is important. If we added new predictors only for FT-Transformer, we would no longer know whether a performance difference came from the algorithm or from extra information.

In [4]:
target_col = "readmitted_30"

categorical_features = [
    "gender",
    "race_group",
    "age_group",
    "admission_source_group",
    "discharge_group",
    "medical_specialty_group",
    "primary_diagnosis",
    "hba1c_group",
    "max_glu_serum",
    "diabetesMed"
]

numeric_features = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses"
]

model_features = (
    categorical_features
    + numeric_features
)

missing_features = [
    feature
    for feature in model_features
    if feature not in df.columns
]

if missing_features:
    raise ValueError(
        "These modelling features are missing: "
        f"{missing_features}"
    )

X = df[model_features].copy()
y = df[target_col].astype(int).copy()

print("X shape:")
print(X.shape)

print("\nFeatures used:")
print(X.columns.tolist())

print("\nTarget counts:")
print(y.value_counts())

print("\nTarget proportions:")
print(y.value_counts(normalize=True))

X shape:
(69987, 18)

Features used:
['gender', 'race_group', 'age_group', 'admission_source_group', 'discharge_group', 'medical_specialty_group', 'primary_diagnosis', 'hba1c_group', 'max_glu_serum', 'diabetesMed', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']

Target counts:
readmitted_30
0    63702
1     6285
Name: count, dtype: int64

Target proportions:
readmitted_30
0    0.910198
1    0.089802
Name: proportion, dtype: float64


In [5]:
forbidden_features = {
    "race",
    "age",
    "medical_specialty",
    "diag_1",
    "A1Cresult",
    "admission_type_id",
    "admission_source_id",
    "discharge_disposition_id",
    "readmitted",
    "readmitted_30",
    "encounter_id",
    "patient_nbr"
}

unexpected_features = (
    forbidden_features
    .intersection(X.columns)
)

assert not unexpected_features, (
    "Unexpected or potentially leaking features found: "
    f"{unexpected_features}"
)

assert not X.columns.duplicated().any()
assert len(X) == len(y)
assert y.isna().sum() == 0
assert set(y.unique()).issubset({0, 1})

print("Feature and target checks passed.")

Feature and target checks passed.


## 4. Create model-training, threshold-validation, test, and internal early-stopping sets

The actual final split used here is:

- **60% model-training data**
- **20% threshold-validation data**
- **20% final-test data**

A further internal split is taken from the 60% model-training portion. It is used only during neural-network optimisation and early stopping.

The 20% threshold-validation set is not used to choose the neural-network architecture, learning rate, training duration, class weight, or any other model setting.

In [6]:

# IMPORTANT: the final-test rows are deliberately NOT loaded in this notebook.
# Notebook 10 creates one fixed split that every model reuses.

SPLIT_DIR = PROJECT_ROOT / "Final_Evaluation" / "Data_Splits"

train_split_path = SPLIT_DIR / "model_train_rows.csv"
validation_split_path = SPLIT_DIR / "threshold_validation_rows.csv"

if not train_split_path.exists() or not validation_split_path.exists():
    raise FileNotFoundError(
        "Final split files are missing. Run 10_create_final_split.ipynb first."
    )

train_idx = (
    pd.read_csv(train_split_path)["row_position"]
    .astype(int)
    .to_numpy()
)
val_idx = (
    pd.read_csv(validation_split_path)["row_position"]
    .astype(int)
    .to_numpy()
)

assert set(train_idx).isdisjoint(set(val_idx))

X_model_train = X.iloc[train_idx].copy()
y_model_train = y.iloc[train_idx].copy()

X_val = X.iloc[val_idx].copy()
y_val = y.iloc[val_idx].copy()

split_summary = pd.DataFrame({
    "split": ["model_training", "threshold_validation"],
    "rows": [len(X_model_train), len(X_val)],
    "positive_count": [int(y_model_train.sum()), int(y_val.sum())],
    "positive_rate": [float(y_model_train.mean()), float(y_val.mean())],
})

print(
    "Final-test rows have NOT been loaded. "
    "They stay locked until notebook 17."
)
split_summary

# XGBoost/CatBoost/FT-Transformer need an additional internal holdout
# from the model-training portion for early stopping/training-duration selection.
X_search_train, X_early_stop, y_search_train, y_early_stop = train_test_split(
    X_model_train,
    y_model_train,
    test_size=0.20,
    stratify=y_model_train,
    random_state=42
)

internal_summary = pd.DataFrame({
    "split": [
        "CV/search training",
        "internal early stopping",
        "full model training after selection",
        "threshold validation",
    ],
    "rows": [
        len(X_search_train),
        len(X_early_stop),
        len(X_model_train),
        len(X_val),
    ],
    "positive_rate": [
        float(y_search_train.mean()),
        float(y_early_stop.mean()),
        float(y_model_train.mean()),
        float(y_val.mean()),
    ],
})

internal_summary


Final-test rows have NOT been loaded. They stay locked until notebook 17.


,split,rows,positive_rate
0,CV/search training,33592,0.089813
1,internal early stopping,8399,0.089773
2,full model training after selection,41991,0.089805
3,threshold validation,13998,0.089799


## 5. FT-Transformer preprocessing

FT-Transformer does not use one-hot encoded categorical variables.

Instead:

### Categorical features
Each category is converted to an integer ID and FT-Transformer learns an embedding for it.

For example:

```text
race_group = "Caucasian"
```

might become category ID `3`, and the model learns a small vector representing that category.

ID `0` is reserved for missing or previously unseen categories.

### Numerical features
Numerical features are:

1. median-imputed using values learned from the training data;
2. standardised to approximately mean 0 and standard deviation 1.

Crucially, the preprocessing objects are always fitted only on the relevant training data. Validation and test data never determine category mappings, medians, means, or standard deviations.

In [7]:
class FTTPreprocessor:
    def __init__(
        self,
        categorical_features,
        numeric_features
    ):
        self.categorical_features = list(
            categorical_features
        )
        self.numeric_features = list(
            numeric_features
        )

        self.category_maps = {}
        self.cat_cardinalities = None
        self.numeric_imputer = SimpleImputer(
            strategy="median"
        )
        self.numeric_scaler = StandardScaler()

    @staticmethod
    def _category_series(series):
        values = series.astype("object")
        values = values.where(
            values.notna(),
            "Missing"
        )
        return values.astype(str)

    def fit(self, X_data):
        self.category_maps = {}

        cardinalities = []

        for feature in self.categorical_features:
            values = self._category_series(
                X_data[feature]
            )

            categories = sorted(
                values.unique().tolist()
            )

            # 0 is reserved for unseen / unknown.
            mapping = {
                category: index + 1
                for index, category
                in enumerate(categories)
            }

            self.category_maps[feature] = mapping

            cardinalities.append(
                len(mapping) + 1
            )

        numeric = (
            X_data[self.numeric_features]
            .apply(
                pd.to_numeric,
                errors="coerce"
            )
        )

        numeric_imputed = (
            self.numeric_imputer
            .fit_transform(numeric)
        )

        self.numeric_scaler.fit(
            numeric_imputed
        )

        self.cat_cardinalities = cardinalities

        return self

    def transform(self, X_data):
        categorical_columns = []

        for feature in self.categorical_features:
            values = self._category_series(
                X_data[feature]
            )

            mapping = self.category_maps[
                feature
            ]

            encoded = (
                values
                .map(mapping)
                .fillna(0)
                .astype(np.int64)
                .to_numpy()
            )

            categorical_columns.append(
                encoded
            )

        if categorical_columns:
            x_cat = np.column_stack(
                categorical_columns
            ).astype(np.int64)
        else:
            x_cat = np.empty(
                (len(X_data), 0),
                dtype=np.int64
            )

        numeric = (
            X_data[self.numeric_features]
            .apply(
                pd.to_numeric,
                errors="coerce"
            )
        )

        numeric_imputed = (
            self.numeric_imputer
            .transform(numeric)
        )

        x_cont = (
            self.numeric_scaler
            .transform(numeric_imputed)
            .astype(np.float32)
        )

        return x_cont, x_cat

    def fit_transform(self, X_data):
        self.fit(X_data)
        return self.transform(X_data)

## 6. DataLoader helpers

PyTorch trains in mini-batches rather than fitting the entire dataset in one operation.

A batch size such as 512 means that 512 encounters are passed through the network before one optimiser update is made.

In [8]:
def make_training_loader(
    x_cont,
    x_cat,
    y_values,
    batch_size,
    shuffle
):
    dataset = TensorDataset(
        torch.tensor(
            x_cont,
            dtype=torch.float32
        ),
        torch.tensor(
            x_cat,
            dtype=torch.long
        ),
        torch.tensor(
            np.asarray(y_values),
            dtype=torch.float32
        )
    )

    return DataLoader(
        dataset,
        batch_size=int(batch_size),
        shuffle=shuffle,
        drop_last=False,
        pin_memory=(
            DEVICE.type == "cuda"
        )
    )


def make_prediction_loader(
    x_cont,
    x_cat,
    batch_size
):
    dataset = TensorDataset(
        torch.tensor(
            x_cont,
            dtype=torch.float32
        ),
        torch.tensor(
            x_cat,
            dtype=torch.long
        )
    )

    return DataLoader(
        dataset,
        batch_size=int(batch_size),
        shuffle=False,
        drop_last=False,
        pin_memory=(
            DEVICE.type == "cuda"
        )
    )

## 7. Build the FT-Transformer

The official implementation expects:

- continuous features in one tensor;
- categorical integer IDs in another tensor.

Important architecture settings:

- `n_blocks`: number of Transformer blocks;
- `d_block`: size of each feature-token representation;
- `attention_n_heads`: number of attention heads;
- `attention_dropout`: dropout inside self-attention;
- `ffn_d_hidden_multiplier`: size of the feed-forward part of each Transformer block;
- `ffn_dropout`: dropout in the feed-forward network;
- `residual_dropout`: dropout on residual connections.

The model outputs a **logit**, not a probability. A sigmoid is applied only when probabilities are required.

In [9]:
def build_ft_transformer(
    n_cont_features,
    cat_cardinalities,
    architecture
):
    model = FTTransformer(
        n_cont_features=n_cont_features,
        cat_cardinalities=(
            cat_cardinalities
        ),
        d_out=1,
        n_blocks=int(
            architecture["n_blocks"]
        ),
        d_block=int(
            architecture["d_block"]
        ),
        attention_n_heads=int(
            architecture[
                "attention_n_heads"
            ]
        ),
        attention_dropout=float(
            architecture[
                "attention_dropout"
            ]
        ),
        ffn_d_hidden=None,
        ffn_d_hidden_multiplier=float(
            architecture[
                "ffn_d_hidden_multiplier"
            ]
        ),
        ffn_dropout=float(
            architecture[
                "ffn_dropout"
            ]
        ),
        residual_dropout=float(
            architecture[
                "residual_dropout"
            ]
        )
    )

    return model.to(DEVICE)

## 8. Training and prediction helpers

Binary cross-entropy is used through `BCEWithLogitsLoss`.

`pos_weight` controls how much more strongly the loss penalises mistakes on readmitted patients:

```text
pos_weight = 1
```

means no additional class weighting.

```text
pos_weight = 4
```

makes the positive/readmission part of the loss four times as important.

This can help the minority class, but it can also distort probability calibration. For that reason, it is tuned rather than automatically fixed to the class imbalance ratio.

Early stopping is based on **AUPRC**, not accuracy.

In [10]:
@torch.inference_mode()
def predict_probabilities(
    model,
    x_cont,
    x_cat,
    batch_size
):
    model.eval()

    loader = make_prediction_loader(
        x_cont=x_cont,
        x_cat=x_cat,
        batch_size=batch_size
    )

    probabilities = []

    for batch_cont, batch_cat in loader:
        batch_cont = batch_cont.to(DEVICE)
        batch_cat = batch_cat.to(DEVICE)

        logits = model(
            batch_cont,
            batch_cat
        ).squeeze(-1)

        batch_probability = torch.sigmoid(
            logits
        )

        probabilities.append(
            batch_probability
            .detach()
            .cpu()
            .numpy()
        )

    return np.concatenate(probabilities)


def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion
):
    model.train()

    total_loss = 0.0
    total_rows = 0

    for (
        batch_cont,
        batch_cat,
        batch_y
    ) in loader:

        batch_cont = batch_cont.to(DEVICE)
        batch_cat = batch_cat.to(DEVICE)
        batch_y = batch_y.to(DEVICE)

        optimizer.zero_grad(
            set_to_none=True
        )

        logits = model(
            batch_cont,
            batch_cat
        ).squeeze(-1)

        loss = criterion(
            logits,
            batch_y
        )

        loss.backward()

        # A modest safety limit for unstable trials.
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        rows = len(batch_y)

        total_loss += (
            loss.item() * rows
        )
        total_rows += rows

    return total_loss / total_rows


def fit_with_early_stopping(
    x_train_cont,
    x_train_cat,
    y_train_values,
    x_valid_cont,
    x_valid_cat,
    y_valid_values,
    cat_cardinalities,
    architecture,
    learning_rate,
    weight_decay,
    batch_size,
    pos_weight,
    max_epochs,
    patience,
    seed=RANDOM_SEED
):
    seed_everything(seed)

    model = build_ft_transformer(
        n_cont_features=(
            x_train_cont.shape[1]
        ),
        cat_cardinalities=(
            cat_cardinalities
        ),
        architecture=architecture
    )

    optimizer = torch.optim.AdamW(
        model.make_parameter_groups(),
        lr=float(learning_rate),
        weight_decay=float(weight_decay)
    )

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(
            [float(pos_weight)],
            dtype=torch.float32,
            device=DEVICE
        )
    )

    train_loader = make_training_loader(
        x_cont=x_train_cont,
        x_cat=x_train_cat,
        y_values=y_train_values,
        batch_size=batch_size,
        shuffle=True
    )

    best_state = None
    best_epoch = 0
    best_auprc = -np.inf
    epochs_without_improvement = 0
    history = []

    for epoch in range(
        1,
        int(max_epochs) + 1
    ):
        train_loss = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            criterion=criterion
        )

        valid_proba = predict_probabilities(
            model=model,
            x_cont=x_valid_cont,
            x_cat=x_valid_cat,
            batch_size=batch_size
        )

        valid_auprc = average_precision_score(
            y_valid_values,
            valid_proba
        )

        valid_auroc = roc_auc_score(
            y_valid_values,
            valid_proba
        )

        valid_brier = brier_score_loss(
            y_valid_values,
            valid_proba
        )

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "validation_auprc": valid_auprc,
            "validation_auroc": valid_auroc,
            "validation_brier_score": valid_brier
        })

        if valid_auprc > best_auprc + 1e-6:
            best_auprc = valid_auprc
            best_epoch = epoch

            best_state = deepcopy(
                model.state_dict()
            )

            epochs_without_improvement = 0

        else:
            epochs_without_improvement += 1

        if (
            epochs_without_improvement
            >= int(patience)
        ):
            break

    if best_state is None:
        raise RuntimeError(
            "Training did not create a valid best model."
        )

    model.load_state_dict(best_state)

    history = pd.DataFrame(history)

    return (
        model,
        history,
        int(best_epoch)
    )

## 9. Final fixed-epoch fitting helper

The internal early-stopping set is used to determine how many epochs are useful.

After that number is selected, a fresh FT-Transformer is trained from scratch on the **complete 60% model-training set** for exactly that many epochs.

This is similar to refitting the final XGBoost/CatBoost model after selecting the number of boosting rounds.

In [11]:
def fit_fixed_epochs(
    x_train_cont,
    x_train_cat,
    y_train_values,
    cat_cardinalities,
    architecture,
    learning_rate,
    weight_decay,
    batch_size,
    pos_weight,
    epochs,
    seed=RANDOM_SEED
):
    seed_everything(seed)

    model = build_ft_transformer(
        n_cont_features=(
            x_train_cont.shape[1]
        ),
        cat_cardinalities=(
            cat_cardinalities
        ),
        architecture=architecture
    )

    optimizer = torch.optim.AdamW(
        model.make_parameter_groups(),
        lr=float(learning_rate),
        weight_decay=float(weight_decay)
    )

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(
            [float(pos_weight)],
            dtype=torch.float32,
            device=DEVICE
        )
    )

    train_loader = make_training_loader(
        x_cont=x_train_cont,
        x_cat=x_train_cat,
        y_values=y_train_values,
        batch_size=batch_size,
        shuffle=True
    )

    history = []

    for epoch in range(
        1,
        int(epochs) + 1
    ):
        train_loss = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            criterion=criterion
        )

        history.append({
            "epoch": epoch,
            "train_loss": train_loss
        })

    return (
        model,
        pd.DataFrame(history)
    )

## 10. Evaluation and threshold-selection helpers

These are deliberately the same metrics and threshold logic used for the previous tuned models.

In [12]:
def evaluate_predictions_from_proba(
    y_true,
    y_proba,
    threshold=0.5,
    model_name="Model"
):
    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)

    y_pred = (
        y_proba >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    total = tn + fp + fn + tp
    actual_positive = tp + fn
    actual_negative = tn + fp
    predicted_positive = tp + fp
    predicted_negative = tn + fn

    specificity = (
        tn / actual_negative
        if actual_negative > 0
        else np.nan
    )

    false_positive_rate = (
        fp / actual_negative
        if actual_negative > 0
        else np.nan
    )

    false_negative_rate = (
        fn / actual_positive
        if actual_positive > 0
        else np.nan
    )

    predicted_positive_rate = (
        predicted_positive / total
        if total > 0
        else np.nan
    )

    patients_flagged_per_true_readmission = (
        predicted_positive / tp
        if tp > 0
        else np.nan
    )

    return {
        "model": model_name,
        "threshold": float(threshold),
        "accuracy": accuracy_score(
            y_true,
            y_pred
        ),
        "precision": precision_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "recall": recall_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "specificity": specificity,
        "false_positive_rate": false_positive_rate,
        "false_negative_rate": false_negative_rate,
        "f1": f1_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "f2": fbeta_score(
            y_true,
            y_pred,
            beta=2,
            zero_division=0
        ),
        "auroc": roc_auc_score(
            y_true,
            y_proba
        ),
        "auprc": average_precision_score(
            y_true,
            y_proba
        ),
        "brier_score": brier_score_loss(
            y_true,
            y_proba
        ),
        "true_negative": int(tn),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_positive": int(tp),
        "predicted_positive": int(
            predicted_positive
        ),
        "predicted_negative": int(
            predicted_negative
        ),
        "predicted_positive_rate": (
            predicted_positive_rate
        ),
        "patients_flagged_per_true_readmission_found": (
            patients_flagged_per_true_readmission
        )
    }


def confusion_matrix_from_proba(
    y_true,
    y_proba,
    threshold=0.5
):
    y_pred = (
        np.asarray(y_proba)
        >= threshold
    ).astype(int)

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    )

    return pd.DataFrame(
        cm,
        index=[
            "Actual not readmitted",
            "Actual readmitted"
        ],
        columns=[
            "Predicted not readmitted",
            "Predicted readmitted"
        ]
    )


def threshold_sweep(
    y_true,
    y_proba,
    model_name="Model",
    thresholds=None
):
    if thresholds is None:
        thresholds = np.round(
            np.arange(
                0.01,
                0.951,
                0.01
            ),
            2
        )

    return pd.DataFrame([
        evaluate_predictions_from_proba(
            y_true=y_true,
            y_proba=y_proba,
            threshold=threshold,
            model_name=model_name
        )
        for threshold in thresholds
    ])


def choose_threshold_for_minimum_recall(
    y_true,
    y_proba,
    min_recall=0.80,
    model_name="Model"
):
    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)

    false_positive_rates, recalls, thresholds = roc_curve(
        y_true,
        y_proba,
        drop_intermediate=False
    )

    candidate_table = pd.DataFrame({
        "threshold": thresholds,
        "recall": recalls,
        "false_positive_rate": false_positive_rates,
        "specificity": 1 - false_positive_rates
    })

    candidate_table = candidate_table[
        np.isfinite(
            candidate_table["threshold"]
        )
    ].copy()

    eligible_candidates = candidate_table[
        candidate_table["recall"]
        >= min_recall
    ].copy()

    if eligible_candidates.empty:
        raise ValueError(
            "No threshold achieved recall >= "
            f"{min_recall:.2f}."
        )

    eligible_candidates = (
        eligible_candidates
        .sort_values(
            by=[
                "false_positive_rate",
                "threshold"
            ],
            ascending=[
                True,
                False
            ]
        )
        .reset_index(drop=True)
    )

    selected_threshold = float(
        eligible_candidates.iloc[0][
            "threshold"
        ]
    )

    selected_metrics = (
        evaluate_predictions_from_proba(
            y_true=y_true,
            y_proba=y_proba,
            threshold=selected_threshold,
            model_name=model_name
        )
    )

    return (
        selected_threshold,
        selected_metrics,
        eligible_candidates
    )

# Stage 1: Architecture search

## 11. Search the Transformer architecture using stratified cross-validation

This first stage answers:

> **What size and shape of FT-Transformer appears to generalise best?**

To keep the search computationally reasonable, the optimiser settings are temporarily fixed:

- learning rate = `3e-4`
- weight decay = `1e-5`
- batch size = `512`
- positive-class weight = `1.0`

Each architecture candidate is evaluated using stratified cross-validation and ranked mainly by mean validation AUPRC.

### Suggested workflow

For a first code check:

```python
ARCHITECTURE_SEARCH_ITERATIONS = 2
ARCHITECTURE_CV_FOLDS = 3
```

For the proper experiment, a reasonable compromise is:

```python
ARCHITECTURE_SEARCH_ITERATIONS = 10
ARCHITECTURE_CV_FOLDS = 5
```

The final setting is still much smaller than a research-scale 100-trial neural-network search, but it is suitable for an MSc comparison experiment without allowing this one model to consume the entire project.

In [13]:
RECALL_TARGET = 0.80

# ---------- SMOKE TEST ----------
# ARCHITECTURE_SEARCH_ITERATIONS = 2
# ARCHITECTURE_CV_FOLDS = 3

# ---------- SUGGESTED FINAL RUN ----------
ARCHITECTURE_SEARCH_ITERATIONS = 10
ARCHITECTURE_CV_FOLDS = 5

ARCHITECTURE_MAX_EPOCHS = 30
ARCHITECTURE_PATIENCE = 5

ARCHITECTURE_FIXED_LR = 3e-4
ARCHITECTURE_FIXED_WEIGHT_DECAY = 1e-5
ARCHITECTURE_FIXED_BATCH_SIZE = 512
ARCHITECTURE_FIXED_POS_WEIGHT = 1.0

architecture_param_distributions = {
    "n_blocks": [
        2, 3, 4
    ],
    "d_block": [
        64, 96, 128, 192
    ],
    "attention_n_heads": [
        4, 8
    ],
    "attention_dropout": [
        0.10, 0.20, 0.30
    ],
    "ffn_d_hidden_multiplier": [
        4 / 3,
        2.0
    ],
    "ffn_dropout": [
        0.00, 0.10, 0.20
    ],
    "residual_dropout": [
        0.00, 0.10
    ]
}

architecture_candidates = list(
    ParameterSampler(
        architecture_param_distributions,
        n_iter=(
            ARCHITECTURE_SEARCH_ITERATIONS
        ),
        random_state=RANDOM_SEED
    )
)

architecture_cv = StratifiedKFold(
    n_splits=ARCHITECTURE_CV_FOLDS,
    shuffle=True,
    random_state=RANDOM_SEED
)

print(
    "Architecture candidates:",
    len(architecture_candidates)
)
print(
    "CV folds:",
    architecture_cv.n_splits
)
print(
    "Total FT-Transformer fits:",
    len(architecture_candidates)
    * architecture_cv.n_splits
)

Architecture candidates: 10
CV folds: 5
Total FT-Transformer fits: 50


In [14]:
architecture_results = []
architecture_fold_results = []

X_search_reset = (
    X_search_train
    .reset_index(drop=True)
)
y_search_reset = (
    y_search_train
    .reset_index(drop=True)
)

for candidate_index, architecture in enumerate(
    architecture_candidates,
    start=1
):
    print(
        f"\nCandidate "
        f"{candidate_index}/"
        f"{len(architecture_candidates)}"
    )
    print(architecture)

    fold_auprcs = []
    fold_aurocs = []
    fold_briers = []
    fold_epochs = []

    candidate_start = time.time()

    for fold_index, (
        fold_train_index,
        fold_valid_index
    ) in enumerate(
        architecture_cv.split(
            X_search_reset,
            y_search_reset
        ),
        start=1
    ):
        fold_X_train = (
            X_search_reset
            .iloc[fold_train_index]
            .copy()
        )
        fold_y_train = (
            y_search_reset
            .iloc[fold_train_index]
            .copy()
        )

        fold_X_valid = (
            X_search_reset
            .iloc[fold_valid_index]
            .copy()
        )
        fold_y_valid = (
            y_search_reset
            .iloc[fold_valid_index]
            .copy()
        )

        preprocessor = FTTPreprocessor(
            categorical_features,
            numeric_features
        )

        (
            fold_train_cont,
            fold_train_cat
        ) = preprocessor.fit_transform(
            fold_X_train
        )

        (
            fold_valid_cont,
            fold_valid_cat
        ) = preprocessor.transform(
            fold_X_valid
        )

        (
            fold_model,
            fold_history,
            fold_best_epoch
        ) = fit_with_early_stopping(
            x_train_cont=fold_train_cont,
            x_train_cat=fold_train_cat,
            y_train_values=fold_y_train,
            x_valid_cont=fold_valid_cont,
            x_valid_cat=fold_valid_cat,
            y_valid_values=fold_y_valid,
            cat_cardinalities=(
                preprocessor.cat_cardinalities
            ),
            architecture=architecture,
            learning_rate=(
                ARCHITECTURE_FIXED_LR
            ),
            weight_decay=(
                ARCHITECTURE_FIXED_WEIGHT_DECAY
            ),
            batch_size=(
                ARCHITECTURE_FIXED_BATCH_SIZE
            ),
            pos_weight=(
                ARCHITECTURE_FIXED_POS_WEIGHT
            ),
            max_epochs=(
                ARCHITECTURE_MAX_EPOCHS
            ),
            patience=(
                ARCHITECTURE_PATIENCE
            ),
            seed=(
                RANDOM_SEED
                + candidate_index * 100
                + fold_index
            )
        )

        fold_valid_proba = (
            predict_probabilities(
                model=fold_model,
                x_cont=fold_valid_cont,
                x_cat=fold_valid_cat,
                batch_size=(
                    ARCHITECTURE_FIXED_BATCH_SIZE
                )
            )
        )

        fold_auprc = (
            average_precision_score(
                fold_y_valid,
                fold_valid_proba
            )
        )

        fold_auroc = roc_auc_score(
            fold_y_valid,
            fold_valid_proba
        )

        fold_brier = brier_score_loss(
            fold_y_valid,
            fold_valid_proba
        )

        fold_auprcs.append(fold_auprc)
        fold_aurocs.append(fold_auroc)
        fold_briers.append(fold_brier)
        fold_epochs.append(fold_best_epoch)

        architecture_fold_results.append({
            "candidate": candidate_index,
            "fold": fold_index,
            **architecture,
            "validation_auprc": fold_auprc,
            "validation_auroc": fold_auroc,
            "validation_brier_score": fold_brier,
            "best_epoch": fold_best_epoch
        })

        del fold_model

        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    elapsed_seconds = (
        time.time() - candidate_start
    )

    architecture_results.append({
        "candidate": candidate_index,
        **architecture,
        "mean_validation_auprc": np.mean(
            fold_auprcs
        ),
        "std_validation_auprc": np.std(
            fold_auprcs
        ),
        "mean_validation_auroc": np.mean(
            fold_aurocs
        ),
        "mean_validation_brier_score": np.mean(
            fold_briers
        ),
        "median_best_epoch": int(
            np.median(fold_epochs)
        ),
        "mean_best_epoch": np.mean(
            fold_epochs
        ),
        "elapsed_seconds": elapsed_seconds
    })

architecture_results = pd.DataFrame(
    architecture_results
)

architecture_fold_results = pd.DataFrame(
    architecture_fold_results
)

architecture_results = (
    architecture_results
    .sort_values(
        by=[
            "mean_validation_auprc",
            "mean_validation_auroc",
            "mean_validation_brier_score"
        ],
        ascending=[
            False,
            False,
            True
        ]
    )
    .reset_index(drop=True)
)

architecture_results.to_csv(
    OUTPUT_DIR
    / "ft_transformer_architecture_cv_results.csv",
    index=False
)

architecture_fold_results.to_csv(
    OUTPUT_DIR
    / "ft_transformer_architecture_cv_fold_results.csv",
    index=False
)

architecture_results


Candidate 1/10
{'residual_dropout': 0.0, 'n_blocks': 2, 'ffn_dropout': 0.0, 'ffn_d_hidden_multiplier': 2.0, 'd_block': 96, 'attention_n_heads': 8, 'attention_dropout': 0.1}

Candidate 2/10
{'residual_dropout': 0.0, 'n_blocks': 2, 'ffn_dropout': 0.0, 'ffn_d_hidden_multiplier': 1.3333333333333333, 'd_block': 64, 'attention_n_heads': 8, 'attention_dropout': 0.3}

Candidate 3/10
{'residual_dropout': 0.1, 'n_blocks': 4, 'ffn_dropout': 0.0, 'ffn_d_hidden_multiplier': 2.0, 'd_block': 96, 'attention_n_heads': 4, 'attention_dropout': 0.3}

Candidate 4/10
{'residual_dropout': 0.1, 'n_blocks': 3, 'ffn_dropout': 0.0, 'ffn_d_hidden_multiplier': 2.0, 'd_block': 64, 'attention_n_heads': 4, 'attention_dropout': 0.2}

Candidate 5/10
{'residual_dropout': 0.1, 'n_blocks': 4, 'ffn_dropout': 0.0, 'ffn_d_hidden_multiplier': 2.0, 'd_block': 64, 'attention_n_heads': 4, 'attention_dropout': 0.1}

Candidate 6/10
{'residual_dropout': 0.1, 'n_blocks': 4, 'ffn_dropout': 0.1, 'ffn_d_hidden_multiplier': 1.333333333

,candidate,residual_dropout,n_blocks,ffn_dropout,ffn_d_hidden_multiplier,d_block,attention_n_heads,attention_dropout,mean_validation_auprc,std_validation_auprc,mean_validation_auroc,mean_validation_brier_score,median_best_epoch,mean_best_epoch,elapsed_seconds
0,2,0.0,2,0.0,1.333333,64,8,0.3,0.147432,0.003158,0.630431,0.080226,6,11.4,120.509676
1,3,0.1,4,0.0,2.000000,96,4,0.3,0.147127,0.002444,0.630440,0.080286,6,6.8,208.660539
2,4,0.1,3,0.0,2.000000,64,4,0.2,0.147032,0.002547,0.628476,0.080328,5,6.8,106.019662
3,7,0.1,4,0.2,1.333333,64,4,0.3,0.146938,0.002828,0.629374,0.080287,17,16.0,220.609726
4,6,0.1,4,0.1,1.333333,64,4,0.2,0.146914,0.003018,0.630364,0.080173,8,11.8,173.940783
5,1,0.0,2,0.0,2.000000,96,8,0.1,0.146791,0.002639,0.629230,0.080302,9,8.2,164.884928
6,10,0.1,2,0.2,1.333333,96,4,0.3,0.146594,0.003011,0.629739,0.080237,12,14.2,170.394220
7,9,0.1,4,0.2,1.333333,96,8,0.2,0.146475,0.002841,0.630579,0.080200,5,8.2,284.949268
8,8,0.0,4,0.1,1.333333,64,4,0.2,0.146434,0.003198,0.627911,0.080306,4,8.8,146.498909
9,5,0.1,4,0.0,2.000000,64,4,0.1,0.146062,0.002926,0.627792,0.080458,7,7.6,138.315415


In [15]:
best_architecture_row = (
    architecture_results.iloc[0]
)

selected_architecture = {
    "n_blocks": int(
        best_architecture_row[
            "n_blocks"
        ]
    ),
    "d_block": int(
        best_architecture_row[
            "d_block"
        ]
    ),
    "attention_n_heads": int(
        best_architecture_row[
            "attention_n_heads"
        ]
    ),
    "attention_dropout": float(
        best_architecture_row[
            "attention_dropout"
        ]
    ),
    "ffn_d_hidden_multiplier": float(
        best_architecture_row[
            "ffn_d_hidden_multiplier"
        ]
    ),
    "ffn_dropout": float(
        best_architecture_row[
            "ffn_dropout"
        ]
    ),
    "residual_dropout": float(
        best_architecture_row[
            "residual_dropout"
        ]
    )
}

print("Selected architecture:")
print(
    json.dumps(
        selected_architecture,
        indent=2
    )
)

print(
    "\nCross-validation AUPRC:",
    best_architecture_row[
        "mean_validation_auprc"
    ]
)

Selected architecture:
{
  "n_blocks": 2,
  "d_block": 64,
  "attention_n_heads": 8,
  "attention_dropout": 0.3,
  "ffn_d_hidden_multiplier": 1.3333333333333333,
  "ffn_dropout": 0.0,
  "residual_dropout": 0.0
}

Cross-validation AUPRC: 0.14743234814044556


# Stage 2: Optimiser, imbalance handling, and training duration

## 12. Search learning rate, weight decay, batch size, and positive-class weight

Now the Transformer architecture is fixed.

This stage asks:

> **How should this architecture be trained?**

We vary:

### `learning_rate`
How large each optimiser update is.

### `weight_decay`
AdamW regularisation. Larger values discourage excessively large learned weights.

### `batch_size`
How many encounters are used for each optimiser update.

### `pos_weight`
How strongly readmission cases are weighted in the binary cross-entropy loss.

Every training candidate uses the internal early-stopping set and is selected primarily by AUPRC.

The 20% threshold-validation set is still untouched.

In [16]:
negative_count = (
    y_model_train == 0
).sum()

positive_count = (
    y_model_train == 1
).sum()

imbalance_ratio = (
    negative_count
    / positive_count
)

print(
    "Negative / positive ratio:",
    imbalance_ratio
)

# ---------- SMOKE TEST ----------
# TRAINING_SEARCH_ITERATIONS = 3

# ---------- SUGGESTED FINAL RUN ----------
TRAINING_SEARCH_ITERATIONS = 12

TRAINING_MAX_EPOCHS = 100
TRAINING_PATIENCE = 10

training_param_distributions = {
    "learning_rate": [
        1e-4,
        3e-4,
        5e-4,
        1e-3
    ],
    "weight_decay": [
        1e-6,
        1e-5,
        1e-4
    ],
    "batch_size": [
        256,
        512,
        1024
    ],
    "pos_weight": [
        1.0,
        2.0,
        4.0,
        6.0,
        float(imbalance_ratio)
    ]
}

training_candidates = list(
    ParameterSampler(
        training_param_distributions,
        n_iter=(
            TRAINING_SEARCH_ITERATIONS
        ),
        random_state=RANDOM_SEED
    )
)

print(
    "Training candidates:",
    len(training_candidates)
)

Negative / positive ratio: 10.135242641209228
Training candidates: 12


In [17]:
stage2_preprocessor = FTTPreprocessor(
    categorical_features,
    numeric_features
)

(
    search_train_cont,
    search_train_cat
) = stage2_preprocessor.fit_transform(
    X_search_train
)

(
    early_stop_cont,
    early_stop_cat
) = stage2_preprocessor.transform(
    X_early_stop
)

training_results = []
training_histories = {}

for candidate_index, config in enumerate(
    training_candidates,
    start=1
):
    print(
        f"\nTraining candidate "
        f"{candidate_index}/"
        f"{len(training_candidates)}"
    )
    print(config)

    candidate_start = time.time()

    (
        candidate_model,
        candidate_history,
        best_epoch
    ) = fit_with_early_stopping(
        x_train_cont=search_train_cont,
        x_train_cat=search_train_cat,
        y_train_values=y_search_train,
        x_valid_cont=early_stop_cont,
        x_valid_cat=early_stop_cat,
        y_valid_values=y_early_stop,
        cat_cardinalities=(
            stage2_preprocessor
            .cat_cardinalities
        ),
        architecture=(
            selected_architecture
        ),
        learning_rate=(
            config["learning_rate"]
        ),
        weight_decay=(
            config["weight_decay"]
        ),
        batch_size=(
            config["batch_size"]
        ),
        pos_weight=(
            config["pos_weight"]
        ),
        max_epochs=(
            TRAINING_MAX_EPOCHS
        ),
        patience=(
            TRAINING_PATIENCE
        ),
        seed=(
            RANDOM_SEED
            + candidate_index
        )
    )

    early_stop_proba = (
        predict_probabilities(
            model=candidate_model,
            x_cont=early_stop_cont,
            x_cat=early_stop_cat,
            batch_size=(
                config["batch_size"]
            )
        )
    )

    training_results.append({
        "candidate": candidate_index,
        **config,
        "best_epoch": best_epoch,
        "early_stop_auprc": (
            average_precision_score(
                y_early_stop,
                early_stop_proba
            )
        ),
        "early_stop_auroc": (
            roc_auc_score(
                y_early_stop,
                early_stop_proba
            )
        ),
        "early_stop_brier_score": (
            brier_score_loss(
                y_early_stop,
                early_stop_proba
            )
        ),
        "elapsed_seconds": (
            time.time() - candidate_start
        )
    })

    training_histories[
        candidate_index
    ] = candidate_history

    del candidate_model

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

training_results = pd.DataFrame(
    training_results
)

training_results = (
    training_results
    .sort_values(
        by=[
            "early_stop_auprc",
            "early_stop_auroc",
            "early_stop_brier_score"
        ],
        ascending=[
            False,
            False,
            True
        ]
    )
    .reset_index(drop=True)
)

training_results.to_csv(
    OUTPUT_DIR
    / "ft_transformer_training_search_results.csv",
    index=False
)

training_results


Training candidate 1/12
{'weight_decay': 1e-05, 'pos_weight': 2.0, 'learning_rate': 0.0003, 'batch_size': 256}

Training candidate 2/12
{'weight_decay': 1e-06, 'pos_weight': 10.135242641209228, 'learning_rate': 0.0005, 'batch_size': 256}

Training candidate 3/12
{'weight_decay': 1e-06, 'pos_weight': 2.0, 'learning_rate': 0.0005, 'batch_size': 1024}

Training candidate 4/12
{'weight_decay': 1e-06, 'pos_weight': 2.0, 'learning_rate': 0.0003, 'batch_size': 512}

Training candidate 5/12
{'weight_decay': 1e-05, 'pos_weight': 6.0, 'learning_rate': 0.0003, 'batch_size': 1024}

Training candidate 6/12
{'weight_decay': 1e-06, 'pos_weight': 1.0, 'learning_rate': 0.0003, 'batch_size': 256}

Training candidate 7/12
{'weight_decay': 1e-06, 'pos_weight': 6.0, 'learning_rate': 0.0003, 'batch_size': 256}

Training candidate 8/12
{'weight_decay': 0.0001, 'pos_weight': 4.0, 'learning_rate': 0.0001, 'batch_size': 512}

Training candidate 9/12
{'weight_decay': 0.0001, 'pos_weight': 4.0, 'learning_rate': 

,candidate,weight_decay,pos_weight,learning_rate,batch_size,best_epoch,early_stop_auprc,early_stop_auroc,early_stop_brier_score,elapsed_seconds
0,3,0.000001,2.000000,0.0005,1024,9,0.153049,0.621584,0.083919,42.800175
1,10,0.000010,10.135243,0.0010,512,7,0.152923,0.622547,0.220647,38.813220
2,12,0.000001,6.000000,0.0005,1024,4,0.152786,0.620030,0.137875,30.176821
3,9,0.000100,4.000000,0.0010,512,5,0.152167,0.621696,0.115810,34.362363
4,2,0.000001,10.135243,0.0005,256,5,0.152070,0.620725,0.235674,40.918667
5,8,0.000100,4.000000,0.0001,512,7,0.152011,0.617931,0.118096,38.771020
6,1,0.000010,2.000000,0.0003,256,4,0.151934,0.620348,0.083632,34.876734
7,7,0.000001,6.000000,0.0003,256,4,0.151507,0.619264,0.163285,37.867142
8,11,0.000001,2.000000,0.0005,512,3,0.151337,0.621227,0.086784,29.904297
9,5,0.000010,6.000000,0.0003,1024,4,0.151173,0.619291,0.157506,30.610045


In [18]:
best_training_row = (
    training_results.iloc[0]
)

selected_training_config = {
    "learning_rate": float(
        best_training_row[
            "learning_rate"
        ]
    ),
    "weight_decay": float(
        best_training_row[
            "weight_decay"
        ]
    ),
    "batch_size": int(
        best_training_row[
            "batch_size"
        ]
    ),
    "pos_weight": float(
        best_training_row[
            "pos_weight"
        ]
    ),
    "epochs": int(
        best_training_row[
            "best_epoch"
        ]
    )
}

selected_candidate_number = int(
    best_training_row[
        "candidate"
    ]
)

best_training_history = (
    training_histories[
        selected_candidate_number
    ]
)

best_training_history.to_csv(
    OUTPUT_DIR
    / "ft_transformer_selected_training_history.csv",
    index=False
)

print("Selected training configuration:")
print(
    json.dumps(
        selected_training_config,
        indent=2
    )
)

print(
    "\nInternal early-stop AUPRC:",
    best_training_row[
        "early_stop_auprc"
    ]
)

Selected training configuration:
{
  "learning_rate": 0.0005,
  "weight_decay": 1e-06,
  "batch_size": 1024,
  "pos_weight": 2.0,
  "epochs": 9
}

Internal early-stop AUPRC: 0.1530488392813967


# Stage 3: Refit the final FT-Transformer

## 13. Fit preprocessing on all 60% model-training data

The categorical mappings, numerical imputation values, and standardisation parameters are now learned from the entire model-training set.

The 20% threshold-validation and 20% test sets are only **transformed**, never used to fit preprocessing.

In [19]:
final_preprocessor = FTTPreprocessor(
    categorical_features,
    numeric_features
)

(
    model_train_cont,
    model_train_cat
) = final_preprocessor.fit_transform(
    X_model_train
)

(
    validation_cont,
    validation_cat
) = final_preprocessor.transform(
    X_val
)


print(
    "Continuous training shape:",
    model_train_cont.shape
)

print(
    "Categorical training shape:",
    model_train_cat.shape
)

print(
    "Categorical cardinalities:",
    final_preprocessor.cat_cardinalities
)

Continuous training shape: (41991, 8)
Categorical training shape: (41991, 10)
Categorical cardinalities: [3, 5, 4, 4, 3, 7, 10, 5, 5, 3]


## 14. Train a fresh model on the complete model-training set

The model is trained for the number of epochs chosen in Stage 2.

No early stopping is performed here because using the 20% threshold-validation set to stop neural-network training would leak information from the set that is supposed to be reserved for threshold selection.

In [20]:
(
    best_ft_transformer_model,
    final_training_history
) = fit_fixed_epochs(
    x_train_cont=model_train_cont,
    x_train_cat=model_train_cat,
    y_train_values=y_model_train,
    cat_cardinalities=(
        final_preprocessor
        .cat_cardinalities
    ),
    architecture=selected_architecture,
    learning_rate=(
        selected_training_config[
            "learning_rate"
        ]
    ),
    weight_decay=(
        selected_training_config[
            "weight_decay"
        ]
    ),
    batch_size=(
        selected_training_config[
            "batch_size"
        ]
    ),
    pos_weight=(
        selected_training_config[
            "pos_weight"
        ]
    ),
    epochs=(
        selected_training_config[
            "epochs"
        ]
    ),
    seed=RANDOM_SEED
)

final_training_history.to_csv(
    OUTPUT_DIR
    / "ft_transformer_final_training_history.csv",
    index=False
)

print(
    "Final FT-Transformer trained for",
    selected_training_config["epochs"],
    "epochs."
)

Final FT-Transformer trained for 9 epochs.


# Stage 4: Threshold selection

## 15. Generate validation probabilities

First we examine the probability distribution and the ordinary threshold of 0.5.

Remember that if the selected `pos_weight` is greater than 1, the raw neural-network probabilities may be poorly calibrated even if their ranking ability is useful.

In [21]:
y_val_proba_ftt = (
    predict_probabilities(
        model=best_ft_transformer_model,
        x_cont=validation_cont,
        x_cat=validation_cat,
        batch_size=(
            selected_training_config[
                "batch_size"
            ]
        )
    )
)

ftt_probability_summary = (
    pd.DataFrame({
        "actual_class": np.asarray(y_val),
        "predicted_readmission_probability": (
            y_val_proba_ftt
        )
    })
    .groupby("actual_class")
    ["predicted_readmission_probability"]
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90
        ]
    )
)

ftt_probability_summary.to_csv(
    OUTPUT_DIR
    / "ft_transformer_validation_probability_summary.csv"
)

ftt_probability_summary

,count,mean,std,min,10%,25%,50%,75%,90%,max
actual_class,,,,,,,,,,
0,12741.0,0.160244,0.051426,0.061934,0.090519,0.121539,0.163773,0.198370,0.216115,0.690526
1,1257.0,0.186314,0.057809,0.063585,0.115065,0.151229,0.189232,0.212291,0.240443,0.570359


In [22]:
ftt_val_default_results = (
    evaluate_predictions_from_proba(
        y_true=y_val,
        y_proba=y_val_proba_ftt,
        threshold=0.5,
        model_name=(
            "FT-Transformer validation default"
        )
    )
)

pd.Series(
    ftt_val_default_results
)

model                                          FT-Transformer validation default
threshold                                                                    0.5
accuracy                                                                0.909987
precision                                                               0.285714
recall                                                                  0.001591
specificity                                                             0.999608
false_positive_rate                                                     0.000392
false_negative_rate                                                     0.998409
f1                                                                      0.003165
f2                                                                      0.001986
auroc                                                                   0.633055
auprc                                                                   0.150173
brier_score                 

## 16. Select the threshold reaching at least 80% recall

Exactly as with the tree models, the selected validation threshold is:

> the threshold with the **lowest false-positive rate** among all thresholds that achieve **recall ≥ 0.80**.

In [23]:
(
    ftt_selected_threshold,
    ftt_val_selected_results,
    ftt_eligible_thresholds
) = choose_threshold_for_minimum_recall(
    y_true=y_val,
    y_proba=y_val_proba_ftt,
    min_recall=RECALL_TARGET,
    model_name=(
        "FT-Transformer validation selected"
    )
)

print(
    "Selected FT-Transformer threshold:"
)
print(
    ftt_selected_threshold
)

pd.DataFrame([
    ftt_val_default_results,
    ftt_val_selected_results
])

Selected FT-Transformer threshold:
0.13860438764095306


,model,threshold,accuracy,precision,recall,specificity,false_positive_rate,false_negative_rate,f1,f2,...,auprc,brier_score,true_negative,false_positive,false_negative,true_positive,predicted_positive,predicted_negative,predicted_positive_rate,patients_flagged_per_true_readmission_found
0,FT-Transformer validation default,0.500000,0.909987,0.285714,0.001591,0.999608,0.000392,0.998409,0.003165,0.001986,...,0.150173,0.085533,12736,5,1255,2,7,13991,0.000500,3.500000
1,FT-Transformer validation selected,0.138604,0.393985,0.108898,0.800318,0.353897,0.646103,0.199682,0.191710,0.352587,...,0.150173,0.085533,4509,8232,251,1006,9238,4760,0.659951,9.182903


In [24]:
ftt_eligible_thresholds.to_csv(
    OUTPUT_DIR
    / "ft_transformer_eligible_validation_thresholds.csv",
    index=False
)

ftt_threshold_sweep = threshold_sweep(
    y_true=y_val,
    y_proba=y_val_proba_ftt,
    model_name="FT-Transformer validation"
)

ftt_threshold_sweep.to_csv(
    OUTPUT_DIR
    / "ft_transformer_validation_threshold_sweep.csv",
    index=False
)

ftt_threshold_sweep[
    [
        "threshold",
        "recall",
        "precision",
        "specificity",
        "false_positive_rate",
        "f1",
        "f2",
        "true_positive",
        "false_positive",
        "false_negative",
        "predicted_positive_rate"
    ]
]

,threshold,recall,precision,specificity,false_positive_rate,f1,f2,true_positive,false_positive,false_negative,predicted_positive_rate
0,0.01,1.0,0.089799,0.0,1.0,0.164798,0.330337,1257,12741,0,1.0
1,0.02,1.0,0.089799,0.0,1.0,0.164798,0.330337,1257,12741,0,1.0
2,0.03,1.0,0.089799,0.0,1.0,0.164798,0.330337,1257,12741,0,1.0
3,0.04,1.0,0.089799,0.0,1.0,0.164798,0.330337,1257,12741,0,1.0
4,0.05,1.0,0.089799,0.0,1.0,0.164798,0.330337,1257,12741,0,1.0
...,...,...,...,...,...,...,...,...,...,...,...
90,0.91,0.0,0.000000,1.0,0.0,0.000000,0.000000,0,0,1257,0.0
91,0.92,0.0,0.000000,1.0,0.0,0.000000,0.000000,0,0,1257,0.0
92,0.93,0.0,0.000000,1.0,0.0,0.000000,0.000000,0,0,1257,0.0
93,0.94,0.0,0.000000,1.0,0.0,0.000000,0.000000,0,0,1257,0.0


In [25]:
comparison_columns = [
    "model",
    "threshold",
    "auprc",
    "auroc",
    "brier_score",
    "accuracy",
    "recall",
    "precision",
    "specificity",
    "false_positive_rate",
    "false_negative_rate",
    "f1",
    "f2",
    "true_positive",
    "true_negative",
    "false_positive",
    "false_negative",
    "predicted_positive_rate",
    "patients_flagged_per_true_readmission_found"
]

ftt_validation_comparison = pd.DataFrame([
    ftt_val_default_results,
    ftt_val_selected_results
])[
    comparison_columns
]

ftt_validation_comparison.to_csv(
    OUTPUT_DIR
    / "ft_transformer_validation_results.csv",
    index=False
)

ftt_validation_comparison

,model,threshold,auprc,auroc,brier_score,accuracy,recall,precision,specificity,false_positive_rate,false_negative_rate,f1,f2,true_positive,true_negative,false_positive,false_negative,predicted_positive_rate,patients_flagged_per_true_readmission_found
0,FT-Transformer validation default,0.500000,0.150173,0.633055,0.085533,0.909987,0.001591,0.285714,0.999608,0.000392,0.998409,0.003165,0.001986,2,12736,5,1255,0.000500,3.500000
1,FT-Transformer validation selected,0.138604,0.150173,0.633055,0.085533,0.393985,0.800318,0.108898,0.353897,0.646103,0.199682,0.191710,0.352587,1006,4509,8232,251,0.659951,9.182903


In [26]:
ftt_val_selected_cm = (
    confusion_matrix_from_proba(
        y_true=y_val,
        y_proba=y_val_proba_ftt,
        threshold=(
            ftt_selected_threshold
        )
    )
)

ftt_val_selected_cm.to_csv(
    OUTPUT_DIR
    / "ft_transformer_validation_confusion_matrix.csv"
)

ftt_val_selected_cm

,Predicted not readmitted,Predicted readmitted
Actual not readmitted,4509,8232
Actual readmitted,251,1006


## Freeze final development-stage artifacts

In [27]:

RECALL_TARGETS = [0.70, 0.75, 0.80, 0.85, 0.90]

threshold_rows = []

for recall_target in RECALL_TARGETS:
    selected_threshold, selected_metrics, _ = (
        choose_threshold_for_minimum_recall(
            y_true=y_val,
            y_proba=y_val_proba_ftt,
            min_recall=recall_target,
            model_name="FT-Transformer validation"
        )
    )

    threshold_rows.append({
        "target_recall": recall_target,
        "selected_threshold": selected_threshold,
        "validation_recall": selected_metrics["recall"],
        "validation_precision": selected_metrics["precision"],
        "validation_specificity": selected_metrics["specificity"],
        "validation_false_positive_rate": selected_metrics["false_positive_rate"],
        "validation_f2": selected_metrics["f2"],
        "validation_true_positive": selected_metrics["true_positive"],
        "validation_false_negative": selected_metrics["false_negative"],
        "validation_false_positive": selected_metrics["false_positive"],
        "validation_true_negative": selected_metrics["true_negative"],
        "validation_flagged_rate": selected_metrics["predicted_positive_rate"],
    })

selected_thresholds = pd.DataFrame(threshold_rows)

selected_thresholds.to_csv(
    OUTPUT_DIR / "selected_validation_thresholds.csv",
    index=False
)

validation_predictions = pd.DataFrame({
    "row_position": val_idx,
    "y_true": np.asarray(y_val, dtype=int),
    "probability": np.asarray(y_val_proba_ftt, dtype=float),
})

validation_predictions.to_csv(
    OUTPUT_DIR / "validation_predictions.csv",
    index=False
)

selected_thresholds

# Save the frozen neural model.
torch.save(
    best_ft_transformer_model.state_dict(),
    OUTPUT_DIR / "final_model_state_dict.pt"
)

with open(OUTPUT_DIR / "selected_architecture.json", "w") as f:
    json.dump(selected_architecture, f, indent=2)

with open(OUTPUT_DIR / "selected_training_config.json", "w") as f:
    json.dump(selected_training_config, f, indent=2)

with open(OUTPUT_DIR / "category_mappings.json", "w") as f:
    json.dump(final_preprocessor.category_maps, f, indent=2)

numeric_preprocessing = pd.DataFrame({
    "feature": numeric_features,
    "imputer_median": final_preprocessor.numeric_imputer.statistics_,
    "scaler_mean": final_preprocessor.numeric_scaler.mean_,
    "scaler_scale": final_preprocessor.numeric_scaler.scale_,
})
numeric_preprocessing.to_csv(
    OUTPUT_DIR / "numeric_preprocessing.csv",
    index=False
)

print("Saved frozen FT-Transformer model and preprocessing artifacts.")


Saved frozen FT-Transformer model and preprocessing artifacts.
